# 05 · Compare ground-truth agreement across P01–P12

**Question:** How does agreement vary across these twelve programs, their measured
complexity, and two probe designs?

P01–P12 are program labels, not a complexity ordering. This notebook reuses
[Step 4](04_alignment.ipynb)'s checks. A/B remain focused causal examples there.

## Configure · definitions fixed before this run

**Matched (main comparison):** every unique sequence in the existing seeded
random-and-structured candidate pool, at every position, with equal observation
weights. Defaults give 3,383 sequences × 6 positions = 20,298 observations per
program. This is the entire candidate pool, not every possible vocabulary sequence.

**Balanced (sensitivity comparison):** recreate Step 3/4's program-specific class
balance, seed, row order, and repeated targets. Rerun measurements to add expected
CKA; the retained Step 4 results stay intact. Positions within a sequence and
repeated targets are not independent trials. Output classes need not balance in
the matched design.

| Check | Per-program summary |
| --- | --- |
| PR | Mean absolute observed-minus-expected PR across all post-embedding half-layer stages, equally weighted; retain signed mean and final gap. |
| Lane values | Mean fraction within configured tolerance at each computed variable's first write, equally weighting scalar lanes and observations. Wider categorical variables contribute more lanes. |
| CKA | Mean absolute observed-minus-expected adjacent CKA where both are defined; report defined counts and disagreements about whether CKA is defined. No valid pairs means N/A. |

Expected representations use Step 4's compiler-encoded variables written so far,
with unwritten and scratch lanes zero. We compare computed-variable count,
maximum dependency depth, and compiled transformer layer count separately.
There is no combined score or live-variable estimate.

In [ ]:
import sys
from pathlib import Path

# Run from the repository root or any folder inside it.
start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (root for root in (start, *start.parents) if (root / "raspformer").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the RaspFormer repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
import json
from dataclasses import asdict

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from raspformer.alignment import measure_alignment, summarize_alignment
from raspformer.compiler import CompilerConfig, record_run, write_json
from raspformer.geometry import (
    ProbeConfig, build_balanced_probes, build_matched_probes, collect_activations,
)
from raspformer.programs import build_programs

config = CompilerConfig(rel_tol=1e-4, abs_tol=1e-4)
probe_config = ProbeConfig()
programs = build_programs(config.max_seq_len)  # P01–P12 only, in their original order.
OUTPUT_ROOT = PROJECT_ROOT / "results" / "comparison"
VIEW = "computed"  # Both views are measured; this selects the figures.
COMPLEXITY = ("computed_variable_count", "max_dependency_depth", "transformer_layers")
METRICS = {
    "pr_mae": "PR mean absolute error ↓",
    "lane_agreement": "Lane agreement ↑",
    "cka_mae": "CKA mean absolute error ↓",
}

## Run

One run folder holds both probe designs, exact observations, raw measurements, and
`comparison.csv`. Compiled target outputs must agree with RASP. A completed numerical
run records measurements; it does not require small metric errors. Plotting follows.

In [ ]:
settings = {
    "probe_config": asdict(probe_config),
    "sampling": {
        "matched": "all deduplicated candidates, all positions, equal observation weights; shared row order",
        "balanced": "Step 3 class/position balancing in original program order; repeats retain weight",
    },
    "complexity": list(COMPLEXITY),
    "ground_truth": "compiler-encoded variables written so far; scratch and unwritten lanes zero; full view includes inputs",
    "agreement": {
        "pr": "mean absolute gap over all post-embedding half-layer stages, equal stage weights",
        "lanes": "mean fraction_close at first write, equal scalar-lane and observation weights",
        "cka": "mean absolute adjacent-score gap only where both scores finite; retain coverage and defined-status mismatches",
    },
    "interpretation": "descriptive compiler benchmark; dependent checks; no live-variable count or causal complexity claim",
}
summaries, stage_tables = [], []
with record_run(OUTPUT_ROOT, programs, config, **settings) as (RUN_DIR, evidence):
    for design, sampler in (("matched", build_matched_probes), ("balanced", build_balanced_probes)):
        folder = RUN_DIR / design
        folder.mkdir()
        evidence.update(stage=f"{design}: probes")
        write_json(RUN_DIR / "manifest.json", evidence)
        probes = sampler(programs, config, probe_config)
        probes.audit.to_csv(folder / "class_counts.csv", index=False)
        # Save the design before compilation, including if compilation later fails.
        for name, table in probes.observations.items():
            table.assign(tokens=table.tokens.map(json.dumps)).to_csv(folder / f"probes_{name}.csv", index=False)
        evidence.update(stage=f"{design}: activations")
        write_json(RUN_DIR / "manifest.json", evidence)
        data = collect_activations(programs, config, probes)
        for name, program in programs.items():
            evidence.update(stage=f"{design}: alignment", current_program=name)
            write_json(RUN_DIR / "manifest.json", evidence)
            tables = measure_alignment(name, program, config, data)
            for key, frame in tables.items():
                file = folder / f"{key}.csv"
                frame.to_csv(file, mode="a", header=not file.exists(), index=False)
            summary = summarize_alignment(tables["stages"], tables["lanes"]).assign(design=design)
            summaries.append(summary)
            stage_tables.append(tables["stages"].assign(design=design))
            summary_file = RUN_DIR / "comparison.csv"
            summary.to_csv(summary_file, mode="a", header=not summary_file.exists(), index=False)
            evidence["programs"].append({"design": design, "program": name, "observations": len(data.observations[name])})
            print(f"{design} / {name}: agreement recorded")
        del data  # Release full residual arrays before collecting the other design.
comparison = pd.concat(summaries, ignore_index=True)
stages = pd.concat(stage_tables, ignore_index=True)
FIGURE_DIR = RUN_DIR / "figures"
FIGURE_DIR.mkdir()
print(f"Saved comparison: {RUN_DIR}")

## Inspect · agreement and coverage

The table displays N/A for undefined CKA agreement. `cka_paired_defined` is the
number of transitions entering its mean; compare it with `transitions` and
`cka_definition_mismatches`. An observed scratch signal can vary while the ideal
representation is still constant. Lane agreement is identical between full and
computed views because it checks the same named variable lanes.

In [ ]:
selected = comparison[comparison.view == VIEW]
display(selected[[
    "design", "program", "observations", *COMPLEXITY, *METRICS,
    "first_write_lanes", "transitions", "cka_observed_defined", "cka_expected_defined",
    "cka_paired_defined", "cka_definition_mismatches",
]].round(5).fillna("N/A"))

## Inspect · complexity comparison

Main figures use matched probes. Labels identify programs; overlapping values can
hide points, so consult the table. PR and CKA errors use a symmetric log scale
(linear near zero) to keep tiny numerical differences visible. Lane agreement
uses a zoomed linear scale. Nearby point labels are grouped; the points and saved
tables retain exact values. No trend is fitted across these twelve selected programs.

In [ ]:
main = selected[selected.design == "matched"]
fig, axes = plt.subplots(3, 3, figsize=(15, 11), constrained_layout=True)
for row_axes, (metric, label) in zip(axes, METRICS.items()):
    unavailable = main.loc[main[metric].isna(), "program"].str.split("_").str[0].tolist()
    for ax, complexity in zip(row_axes, COMPLEXITY):
        valid = main.dropna(subset=[metric])
        ax.scatter(valid[complexity], valid[metric], s=35)
        # Group nearby labels only; all scatter points retain their exact values.
        for (x, _), points in valid.groupby([complexity, valid[metric].round(5)]):
            ids = points.program.str.split("_").str[0].tolist()
            text = "\n".join(", ".join(ids[i:i + 3]) for i in range(0, len(ids), 3))
            ax.annotate(text, (x, points[metric].mean()), xytext=(4, 4),
                        textcoords="offset points", fontsize=7)
        ax.margins(x=0.25, y=0.2)
        ax.set(xlabel=complexity.replace("_", " "), ylabel=label)
        if metric != "lane_agreement":
            ax.set_yscale("symlog", linthresh=1e-8)
            ax.set_ylim(bottom=0)
        else:
            ax.set_ylim(max(0, main[metric].min() - 0.02), 1.04)
        if unavailable:
            ax.text(0.02, 0.97, "N/A: " + ", ".join(unavailable), transform=ax.transAxes,
                    va="top", fontsize=8)
        ax.grid(alpha=0.2)
fig.suptitle(f"P01–P12 · matched probes · {VIEW}")
fig.savefig(FIGURE_DIR / f"complexity_{VIEW}.png", bbox_inches="tight")
plt.show()
plt.close(fig)

## Inspect · sensitivity to the probe design

Compare both designs within each program. The paired table records
**matched minus balanced** for each measure. A difference is a sampling sensitivity,
not an estimate of statistical significance. CKA summaries may average different
sets of defined transitions; inspect coverage before interpreting their difference.

In [ ]:
paired = selected.pivot(index="program", columns="design", values=list(METRICS))
for metric in METRICS:
    paired[(metric, "matched_minus_balanced")] = paired[(metric, "matched")] - paired[(metric, "balanced")]
paired = paired.sort_index(axis=1, level=0, sort_remaining=False)
paired.to_csv(RUN_DIR / f"sampling_comparison_{VIEW}.csv")
display(paired.round(6).fillna("N/A"))
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
labels = [name.split("_")[0] for name in programs]
for ax, (metric, label) in zip(axes, METRICS.items()):
    for design, marker in (("matched", "o"), ("balanced", "x")):
        values = selected[selected.design == design].set_index("program").reindex(programs)[metric]
        ax.scatter(range(len(labels)), values, marker=marker, label=design)
    ax.set(xticks=range(len(labels)), xticklabels=labels, ylabel=label)
    ax.tick_params(axis="x", rotation=60)
    if metric != "lane_agreement":
        ax.set_yscale("symlog", linthresh=1e-8)
        ax.set_ylim(bottom=0)
    else:
        ax.set_ylim(max(0, selected[metric].min() - 0.02), 1.01)
    ax.grid(alpha=0.2)
axes[0].legend()
fig.suptitle(f"Probe-design sensitivity · {VIEW}; missing points = undefined")
fig.savefig(FIGURE_DIR / f"sampling_{VIEW}.png", bbox_inches="tight")
plt.show()
plt.close(fig)

## Inspect · writes and CKA

These stage-level summaries separate scheduled writes from transitions without a
new variable. They show observed and expected CKA and valid-pair counts; a write is
not required to lower CKA, and CKA is not a direct computation detector.
The observed and expected means can use different sets of defined transitions;
their difference is not the paired CKA error.


In [ ]:
transitions = stages[(stages.view == VIEW) & (stages.step > 0)].copy()
transitions["write"] = transitions.new_variable_count > 0
transitions["cka_abs_error"] = (transitions.adjacent_cka - transitions.expected_adjacent_cka).abs()
write_summary = transitions.groupby(["design", "program", "write"], sort=False).agg(
    transitions=("step", "size"),
    observed_defined=("adjacent_cka", "count"), expected_defined=("expected_adjacent_cka", "count"),
    paired_defined=("cka_abs_error", "count"), cka_mae=("cka_abs_error", "mean"),
    observed_mean=("adjacent_cka", "mean"), expected_mean=("expected_adjacent_cka", "mean"),
)
write_summary.to_csv(RUN_DIR / f"cka_writes_{VIEW}.csv")
display(write_summary.round(5).fillna("N/A"))

## Interpret

PR, lane, and CKA agreement share the same underlying representations. Close lane
values can imply close geometric summaries; these are not independent validations
of an interpretability method. Errors can reflect scratch activity, encoding,
finite sampling, or numerical approximation. Shared inputs remove one source of
variation, while the programs and their encodings still differ.

These plots describe this compiler revision and probe design. They do not isolate a
causal effect of complexity or establish a universal trust curve for trained models.
Consult [Step 4](04_alignment.ipynb) for A/B intervention evidence. Numerical
completion excludes the notebook's later plots and summary exports.

In [ ]:
print(f"Numerical status: {evidence['status']}")
print(f"Programs: {comparison.program.nunique()} | Designs: {comparison.design.nunique()}")
print(f"Figures: {len(list(FIGURE_DIR.glob('*.png')))}")
print(f"Results: {RUN_DIR}")